# 🛡️ Defensive Midfielder Profiling — Serie A 2025/26

## Methodology
4-pillar weighted KPI framework from notebook 11, applied exclusively to the **2025/26 season** with three analyst-driven refinements.

| Pillar | Weight | KPIs |
|--------|--------|------|
| **Possession Regain** | 0.40 | Tackles Won p90, Pass Cuts p90 *(interceptions + blocked passes)*, Ball Recoveries p90 |
| **Positional Screening** | 0.25 | Clearances p90, Aerial Win % |
| **Counter-pressing** | 0.20 | Ball Rec. Opp Half p90, Tackles Opp Half p90, Fast Regain % (≤8 passes) |
| **Distribution Security** | 0.15 | Pass Completion %, Short Pass %, Turnovers p90 (neg.) |

$$S = 0.40 \cdot R_{\text{regain}} + 0.25 \cdot R_{\text{screen}} + 0.20 \cdot R_{\text{press}} + 0.15 \cdot R_{\text{dist}}$$

### What's different in this notebook
- **Possession-adjusted** per-90 metrics: count-based KPIs scaled by `team_poss% / league_avg_poss%` so players on low-possession teams are not inflated
- **Pass Cuts**: interceptions + blocked passes merged into a single KPI (both measure reading/cutting passing lanes)
- **Fast Regain ≤8 passes**: realistic counter-pressing window — % of ball regains that happen within 8 opponent passes
- **Interactive charts** (Plotly) with dropdown selectors
- **Single season** focus: Serie A 2025/26 only

In [2]:
# ── Imports & Configuration ───────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os, glob, warnings
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW  = os.path.join(BASE, "data", "raw")
SEASON_TAG  = "serie_a_2025_2026"
SEASON_LABEL = "2025_2026"

DM_POSITIONS = {"CDM", "MC"}
MIN_MINUTES  = 900

PILLAR_WEIGHTS = {
    "Possession Regain":     0.40,
    "Positional Screening":  0.25,
    "Counter-pressing":      0.20,
    "Distribution Security": 0.15,
}

# Interceptions + blocked passes merged into "pass_cuts"
KPI_MAP = {
    "Possession Regain": {
        "tackles_won_p90":     {"weight": 0.35, "negative": False},
        "pass_cuts_p90":       {"weight": 0.40, "negative": False},
        "ball_recoveries_p90": {"weight": 0.25, "negative": False},
    },
    "Positional Screening": {
        "clearances_p90":      {"weight": 0.55, "negative": False},
        "aerial_win_pct":      {"weight": 0.45, "negative": False},
    },
    "Counter-pressing": {
        "ball_rec_opp_half_p90": {"weight": 0.50, "negative": False},
        "tackles_opp_half_p90":  {"weight": 0.50, "negative": False},
    },
    "Distribution Security": {
        "pass_completion_pct": {"weight": 0.33, "negative": False},
        "short_pass_pct":      {"weight": 0.33, "negative": False},
        "turnovers_p90":       {"weight": 0.34, "negative": True},
    },
}

KPI_LABELS = {
    "tackles_won_p90":       "Tackles Won",
    "pass_cuts_p90":         "Pass Cuts (Int.+Blk.)",
    "ball_recoveries_p90":   "Ball Recoveries",
    "clearances_p90":        "Clearances",
    "aerial_win_pct":        "Aerial Win %",
    "ball_rec_opp_half_p90": "Ball Rec. Opp Half",
    "tackles_opp_half_p90":  "Tackles Opp Half",
    "pass_completion_pct":   "Pass Completion %",
    "short_pass_pct":        "Short Pass %",
    "turnovers_p90":         "Turnovers (neg.)",
    "fast_regain_pct":       "Fast Regain %",
}

FAST_REGAIN_THRESHOLD = 8  # ≤8 opponent passes = fast regain

print(f"Target season: {SEASON_TAG}")
print(f"Events dir exists: {os.path.isdir(os.path.join(RAW, SEASON_TAG, 'events'))}")
n_files = len(glob.glob(os.path.join(RAW, SEASON_TAG, "events", "*.csv")))
print(f"Match files: {n_files}")
print(f"Fast regain threshold: ≤{FAST_REGAIN_THRESHOLD} opp. passes")
print("Setup complete ✓")

Target season: serie_a_2025_2026
Events dir exists: True
Match files: 319
Fast regain threshold: ≤8 opp. passes
Setup complete ✓


In [3]:
# ── Load 2025/26 events ──────────────────────────────────────────────────────
evt_dir = os.path.join(RAW, SEASON_TAG, "events")
files = sorted(glob.glob(os.path.join(evt_dir, "*.csv")))
print(f"Loading {len(files)} match files...")

frames = []
for f in files:
    try:
        df = pd.read_csv(f, low_memory=False)
        df["season"] = SEASON_LABEL
        df["gameweek"] = os.path.basename(f).split("_")[0]
        frames.append(df)
    except:
        continue

all_events = pd.concat(frames, ignore_index=True)
dm_events  = all_events[all_events["position"].isin(DM_POSITIONS)].copy()

print(f"Total events: {len(all_events):,}")
print(f"DM/CM events: {len(dm_events):,}")
print(f"Unique DM/CM players: {dm_events['player_name'].nunique()}")

Loading 319 match files...
Total events: 532,428
DM/CM events: 100,436
Unique DM/CM players: 192
Total events: 532,428
DM/CM events: 100,436
Unique DM/CM players: 192


In [4]:
# ── Estimate minutes & compute raw KPIs ──────────────────────────────────────

# --- Minutes ---
dm = dm_events.copy()
dm["time_min"] = pd.to_numeric(dm["time_min"], errors="coerce")

player_match = (
    dm.groupby(["player_name", "player_id", "team_name", "season", "match_id"])
    ["time_min"].max().reset_index().rename(columns={"time_min": "last_event_min"})
)
minutes_df = (
    player_match.groupby(["player_name", "player_id", "team_name", "season"])
    .agg(matches=("match_id", "nunique"), total_minutes=("last_event_min", "sum"))
    .reset_index()
)

# --- Raw KPIs ---
df = dm_events.copy()
for c in ["type_id", "outcome", "x"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

grp = ["player_name", "player_id", "team_name", "season"]

def _cnt(mask):
    return df[mask].groupby(grp).size()

kpis = pd.DataFrame()
kpis["tackles_won"]       = _cnt((df["type_id"]==7) & (df["outcome"]==1))
kpis["interceptions"]     = _cnt(df["type_id"]==8)
kpis["blocked_passes"]    = _cnt(df["type_id"]==74)
kpis["ball_recoveries"]   = _cnt(df["type_id"]==49)
kpis["clearances"]        = _cnt(df["type_id"]==12)
kpis["aerials_won"]       = _cnt((df["type_id"]==44) & (df["outcome"]==1))
kpis["aerials_total"]     = _cnt(df["type_id"]==44)
kpis["ball_rec_opp_half"] = _cnt((df["type_id"]==49) & (df["x"]>50))
kpis["tackles_opp_half"]  = _cnt((df["type_id"]==7) & (df["outcome"]==1) & (df["x"]>50))
kpis["passes_completed"]  = _cnt((df["type_id"]==1) & (df["outcome"]==1))
kpis["passes_total"]      = _cnt(df["type_id"]==1)
kpis["short_passes"]      = _cnt((df["type_id"]==1) & (df["outcome"]==1) & (df["Long ball"].isin(["N/A", np.nan])))
kpis["turnovers"]         = _cnt((df["type_id"]==50) | ((df["type_id"]==61) & (df["outcome"]==0)))
kpis["fouls_committed"]   = _cnt((df["type_id"]==4) & (df["outcome"]==0))
kpis = kpis.fillna(0).reset_index()

# ── Merged KPI: Pass Cuts = Interceptions + Blocked Passes ───
kpis["pass_cuts"] = kpis["interceptions"] + kpis["blocked_passes"]

# --- Merge & per-90 ---
merged = kpis.merge(minutes_df[["player_name","player_id","season","matches","total_minutes"]],
                     on=["player_name","player_id","season"], how="left")
player_stats = merged[merged["total_minutes"] >= MIN_MINUTES].copy()

per90_cols = ["tackles_won","pass_cuts","interceptions","ball_recoveries","blocked_passes",
              "clearances","ball_rec_opp_half","tackles_opp_half","turnovers","fouls_committed"]
for c in per90_cols:
    player_stats[f"{c}_p90"] = (player_stats[c] / player_stats["total_minutes"]) * 90

player_stats["pass_completion_pct"] = (player_stats["passes_completed"]/player_stats["passes_total"]*100).fillna(0)
player_stats["short_pass_pct"]      = (player_stats["short_passes"]/player_stats["passes_completed"]*100).fillna(0)
player_stats["aerial_win_pct"]      = (player_stats["aerials_won"]/player_stats["aerials_total"]*100).fillna(0)

print(f"Players with ≥{MIN_MINUTES} min: {len(player_stats)}")
player_stats.sort_values("pass_cuts_p90", ascending=False)[
    ["player_name","team_name","matches","total_minutes",
     "pass_cuts_p90","interceptions_p90","blocked_passes_p90","tackles_won_p90","ball_recoveries_p90"]
].head(10).round(2)

Players with ≥900 min: 73


,player_name,team_name,matches,total_minutes,pass_cuts_p90,interceptions_p90,blocked_passes_p90,tackles_won_p90,ball_recoveries_p90
161,Y. Ramadani,US Lecce,31,2730,2.41,1.29,1.12,1.35,5.54
84,M. Folorunsho,Cagliari Calcio,15,1084,2.24,0.42,1.83,0.66,3.15
49,J. Akpa Akpro,Hellas Verona FC,12,1004,2.15,0.72,1.43,1.34,3.94
112,N. Pisilli,AS Roma,12,1051,2.14,1.20,0.94,0.77,4.45
35,G. Gineitis,Torino FC,18,1278,2.04,0.92,1.13,1.41,3.31
119,P. Masini,Genoa CFC,20,1825,1.97,1.04,0.94,1.33,3.16
1,A. Bernede,Hellas Verona FC,20,1575,1.89,0.46,1.43,1.37,5.43
151,T. Pobega,Bologna FC 1909,17,1399,1.87,1.09,0.77,0.90,2.25
62,K. Thorstvedt,US Sassuolo Calcio,23,2090,1.77,0.86,0.90,0.47,2.93
89,M. Keita,Parma Calcio 1913,22,1835,1.72,0.98,0.74,1.13,3.83


## ⚖️ Possession-Based Normalization

**Problem**: A CDM at Lecce (~42% possession) defends far more than one at Inter (~58%). Raw per-90 counts inflate low-possession team players.

**Solution — Possession Adjustment Factor**:
1. For each team, compute **average possession %** across all matches
2. `adjustment_factor = team_possession% / league_avg_possession%`
   - Lecce ~42% poss → factor ≈ 0.84 (scales down)
   - Inter ~58% poss → factor ≈ 1.16 (scales up)
3. Applied **only to count-based p90 metrics** — NOT to percentage metrics (pass completion %, aerial win %) which are already ratios

In [5]:
# ── Possession-Based Normalization ──────────────────────────────────────────────
# Compute team possession % as adjustment factor.

ae = all_events.copy()
ae["type_id"] = pd.to_numeric(ae["type_id"], errors="coerce")
ae["outcome"] = pd.to_numeric(ae["outcome"], errors="coerce")

# Successful passes by each team in each match
team_passes = (
    ae[(ae["type_id"] == 1) & (ae["outcome"] == 1)]
    .groupby(["match_id", "team_name"]).size()
    .reset_index(name="passes")
)

# For each match, compute possession % for each team
match_teams = team_passes.groupby("match_id")["team_name"].apply(list).reset_index()
match_teams = match_teams[match_teams["team_name"].apply(len) == 2]

poss_rows = []
for _, r in match_teams.iterrows():
    mid = r["match_id"]
    t1, t2 = r["team_name"]
    mp = team_passes[team_passes["match_id"] == mid].set_index("team_name")["passes"]
    total = mp.get(t1, 0) + mp.get(t2, 0)
    if total > 0:
        poss_rows.append({"team_name": t1, "match_id": mid, "poss_pct": mp.get(t1, 0) / total * 100})
        poss_rows.append({"team_name": t2, "match_id": mid, "poss_pct": mp.get(t2, 0) / total * 100})

poss_df = pd.DataFrame(poss_rows)
team_poss = poss_df.groupby("team_name")["poss_pct"].mean().reset_index(name="avg_possession")

league_avg_poss = team_poss["avg_possession"].mean()
team_poss["adj_factor"] = team_poss["avg_possession"] / league_avg_poss

print(f"League-avg possession: {league_avg_poss:.1f}%")
print(f"\nPossession adjustment factors:")
print(team_poss.sort_values("avg_possession", ascending=False).to_string(index=False))

# ── Apply adjustment to count-based p90 metrics ──────────────────────────────
player_stats = player_stats.merge(team_poss[["team_name", "avg_possession", "adj_factor"]], on="team_name", how="left")

count_based_p90 = [
    "tackles_won_p90", "pass_cuts_p90", "interceptions_p90", "ball_recoveries_p90",
    "blocked_passes_p90", "clearances_p90",
    "ball_rec_opp_half_p90", "tackles_opp_half_p90",
    "turnovers_p90", "fouls_committed_p90",
]

for c in count_based_p90:
    player_stats[f"{c}_raw"] = player_stats[c]
    player_stats[c] = player_stats[c] * player_stats["adj_factor"]

print(f"\nAdjustment effect (pass_cuts_p90):")
player_stats.sort_values("pass_cuts_p90", ascending=False)[
    ["player_name", "team_name", "avg_possession", "adj_factor", "pass_cuts_p90_raw", "pass_cuts_p90"]
].head(10).round(2)

League-avg possession: 50.0%

Possession adjustment factors:
                 team_name  avg_possession  adj_factor
          Calcio Como 1907       63.459001    1.269374
  FC Internazionale Milano       60.950094    1.219188
                SSC Napoli       60.053747    1.201258
               Juventus FC       58.044976    1.161077
                   AS Roma       57.811283    1.156402
           Bologna FC 1909       56.672240    1.133618
Atalanta Bergamasca Calcio       56.052767    1.121226
                  AC Milan       53.280474    1.065772
            ACF Fiorentina       52.542516    1.051011
                  SS Lazio       51.072071    1.021597
                 Genoa CFC       47.623704    0.952619
           Cagliari Calcio       45.203092    0.904200
        US Sassuolo Calcio       44.759595    0.895328
              US Cremonese       44.613717    0.892410
            Udinese Calcio       44.557871    0.891293
                 Torino FC       43.145924    0.863050
    

,player_name,team_name,avg_possession,adj_factor,pass_cuts_p90_raw,pass_cuts_p90
53,N. Pisilli,AS Roma,57.81,1.16,2.14,2.48
69,T. Pobega,Bologna FC 1909,56.67,1.13,1.87,2.11
36,M. Folorunsho,Cagliari Calcio,45.20,0.90,2.24,2.03
71,Y. Ramadani,US Lecce,40.16,0.80,2.41,1.93
56,P. Masini,Genoa CFC,47.62,0.95,1.97,1.88
41,M. Locatelli,Juventus FC,58.04,1.16,1.56,1.81
17,H. Mkhitaryan,FC Internazionale Milano,60.95,1.22,1.47,1.79
15,G. Gineitis,Torino FC,43.15,0.86,2.04,1.76
72,Éderson,Atalanta Bergamasca Calcio,56.05,1.12,1.56,1.75
44,M. Perrone,Calcio Como 1907,63.46,1.27,1.36,1.73


In [6]:
# ── Fast Regain % computation ──────────────────────────────────────────────────

def compute_passes_to_regain(all_events, dm_positions):
    ev = all_events.copy()
    for c in ["type_id","outcome","time_min","time_sec","x","period_id"]:
        ev[c] = pd.to_numeric(ev[c], errors="coerce")
    
    play_types = {1,2,3,4,7,8,9,10,11,12,13,14,15,16,41,44,45,49,50,51,54,61,74}
    ev = ev[ev["type_id"].isin(play_types)].dropna(subset=["match_id","team_name","period_id","time_min"]).copy()
    ev["t_sec"] = ev["time_min"]*60 + ev["time_sec"].fillna(0)
    ev = ev.sort_values(["match_id","period_id","t_sec","event_id"]).reset_index(drop=True)
    
    possession_mask = (
        ((ev["type_id"]==1)&(ev["outcome"]==1)) | (ev["type_id"]==49) |
        ((ev["type_id"]==7)&(ev["outcome"]==1)) | (ev["type_id"]==8) |
        (ev["type_id"]==11) | (ev["type_id"]==52)
    )
    ev["pos_team"] = np.where(possession_mask, ev["team_name"], np.nan)
    ev["pos_team"] = ev.groupby(["match_id","period_id"])["pos_team"].ffill()
    ev["prev_pos_team"] = ev.groupby(["match_id","period_id"])["pos_team"].shift(1)
    ev["pos_change"] = (ev["pos_team"]!=ev["prev_pos_team"]) & ev["prev_pos_team"].notna() & ev["pos_team"].notna()
    ev["is_opp_pass"] = (ev["type_id"]==1)&(ev["outcome"]==1)
    
    regain_types = {7,8,49}
    results = []
    for match_id, mev in ev.groupby("match_id"):
        mev = mev.reset_index(drop=True)
        mr = mev[mev["type_id"].isin(regain_types) & mev["pos_change"] &
                 mev["position"].isin(dm_positions) & ((mev["type_id"]!=7)|(mev["outcome"]==1))]
        if mr.empty: continue
        for idx, r in mr.iterrows():
            prec = mev[(mev.index<idx)&(mev["period_id"]==r["period_id"])]
            lc = prec[prec["pos_change"]&(prec["pos_team"]==r["prev_pos_team"])]
            if lc.empty: continue
            li = lc.index[-1]
            opp_p = len(mev[(mev.index>li)&(mev.index<=idx)&(mev["team_name"]==r["prev_pos_team"])&mev["is_opp_pass"]])
            results.append({
                "match_id": match_id, "season": r["season"],
                "regain_player": r["player_name"], "regain_player_id": r["player_id"],
                "opp_passes_before_regain": opp_p,
            })
    return pd.DataFrame(results)

print("Computing fast regain % (this may take a few minutes)...")
ptr_raw = compute_passes_to_regain(all_events, DM_POSITIONS)
print(f"Regain events: {len(ptr_raw):,}")

# Aggregate — only fast_regain_pct
ptr_agg = (
    ptr_raw.groupby(["regain_player","regain_player_id","season"])
    .agg(n_regains=("opp_passes_before_regain","count"),
         fast_regains=("opp_passes_before_regain", lambda x: (x<=FAST_REGAIN_THRESHOLD).sum()))
    .reset_index()
)
ptr_agg["fast_regain_pct"] = (ptr_agg["fast_regains"]/ptr_agg["n_regains"]*100).round(1)
ptr_agg = ptr_agg[ptr_agg["n_regains"]>=10]
print(f"Players with fast regain data: {len(ptr_agg)}")
ptr_agg.sort_values("fast_regain_pct", ascending=False).head(10)

Computing fast regain % (this may take a few minutes)...
Regain events: 4,853
Players with fast regain data: 105
Regain events: 4,853
Players with fast regain data: 105


,regain_player,regain_player_id,season,n_regains,fast_regains,fast_regain_pct
108,M. Vecino,9462218yklkts6mod3cbfryz9,2025_2026,22,21,95.5
118,N. Rovella,79e6khfyg3rdauul0yjlixv8q,2025_2026,21,20,95.2
152,T. Pobega,ch5rw8dgb6wh99wyhd8o3t1pl,2025_2026,47,44,93.6
111,N. El Aynaoui,6zf7dbi1xepyxdohsnrunu5m,2025_2026,31,29,93.5
151,T. Koopmeiners,2ugzqrpaooto1r24a6kvqf2ll,2025_2026,28,26,92.9
65,J. Vandeputte,d94867tqgcdqdwg7rhhzt1al1,2025_2026,27,25,92.6
31,E. Elmas,72im0m7d8rqpsclo4f5crqmsp,2025_2026,27,25,92.6
8,A. Jashari,94sln58zt76ydfvs4pl7xkqy2,2025_2026,13,12,92.3
138,S. Lobotka,9y8fnvazj02vy31ljx3du0d79,2025_2026,62,57,91.9
68,K. De Bruyne,c2jqa9vozhqu7x79h5rr2n1ed,2025_2026,12,11,91.7


In [7]:
# ── Percentiles, Pillar Scores, Composite Score ──────────────────────────────

# Merge fast regain into player_stats
ptr_merge = ptr_agg[["regain_player","regain_player_id","season",
                      "n_regains","fast_regain_pct"]].rename(
    columns={"regain_player":"player_name","regain_player_id":"player_id"})

ps = player_stats.merge(ptr_merge, on=["player_name","player_id","season"], how="left")

# ── Percentile ranks ─────────────────────────────────────────
for pillar, kpis in KPI_MAP.items():
    for kpi, info in kpis.items():
        pc = f"{kpi}_pct"
        ps[pc] = ps[kpi].rank(pct=True) * 100
        if info["negative"]:
            ps[pc] = 100 - ps[pc]

# Fast regain percentile (higher = better)
ps["fast_regain_pct_pct"] = ps["fast_regain_pct"].rank(pct=True) * 100

# ── Pillar scores (from KPI_MAP) ─────────────────────────────
for pillar, kpis in KPI_MAP.items():
    col = f"pillar_{pillar.replace(' ','_').lower()}"
    ps[col] = sum(ps[f"{k}_pct"] * v["weight"] for k, v in kpis.items())

# ── Enhanced counter-pressing pillar with Fast Regain % ──────
ps["pillar_counter-pressing_v2"] = (
    ps["ball_rec_opp_half_p90_pct"] * 0.35 +
    ps["tackles_opp_half_p90_pct"]  * 0.35 +
    ps["fast_regain_pct_pct"].fillna(50) * 0.30
)

# ── Composite score ──────────────────────────────────────────
ps["composite_score"] = (
    ps["pillar_possession_regain"]     * 0.40 +
    ps["pillar_positional_screening"]  * 0.25 +
    ps["pillar_counter-pressing_v2"]   * 0.20 +
    ps["pillar_distribution_security"] * 0.15
)
ps = ps.sort_values("composite_score", ascending=False).reset_index(drop=True)
ps["rank"] = range(1, len(ps)+1)

print(f"Final scored players: {len(ps)}")
ps[["rank","player_name","team_name","matches","total_minutes","composite_score",
    "pillar_possession_regain","pillar_positional_screening",
    "pillar_counter-pressing_v2","pillar_distribution_security"]].head(15).round(1)

Final scored players: 73


,rank,player_name,team_name,matches,total_minutes,composite_score,pillar_possession_regain,pillar_positional_screening,pillar_counter-pressing_v2,pillar_distribution_security
0,1,M. Locatelli,Juventus FC,29,2371,83.2,95.5,88.3,73.8,54.4
1,2,M. Perrone,Calcio Como 1907,30,2440,78.5,90.6,66.2,80.0,64.4
2,3,B. Cristante,AS Roma,28,2347,76.1,87.1,79.6,79.8,35.8
3,4,Y. Ramadani,US Lecce,31,2730,74.0,84.0,77.5,78.4,35.8
4,5,Éderson,Atalanta Bergamasca Calcio,23,1907,73.2,91.7,63.3,70.8,43.6
5,6,M. de Roon,Atalanta Bergamasca Calcio,26,2154,72.9,71.0,74.5,79.3,66.5
6,7,N. Pisilli,AS Roma,12,1051,72.8,83.5,52.7,90.4,54.4
7,8,M. Koné,AS Roma,26,2376,72.2,82.7,49.4,89.9,58.8
8,9,H. Mkhitaryan,FC Internazionale Milano,21,1713,70.8,84.0,54.2,83.7,45.9
9,10,R. Freuler,Bologna FC 1909,22,1841,70.0,82.2,53.4,80.5,51.5


## 📊 Interactive Composite Ranking

Bar chart of the top 25 players ranked by composite score, with pillar breakdown on hover.

In [8]:
# ── Interactive stacked bar: Top 25 by composite score ────────────────────────

top_n = ps.head(25).copy().sort_values("composite_score", ascending=True)

pillar_info = [
    ("pillar_possession_regain",     "Possession Regain (0.40)",     "#1f77b4"),
    ("pillar_positional_screening",  "Positional Screening (0.25)",  "#ff7f0e"),
    ("pillar_counter-pressing_v2",   "Counter-pressing (0.20)",      "#2ca02c"),
    ("pillar_distribution_security", "Distribution Security (0.15)", "#d62728"),
]

fig_bar = go.Figure()
for col, name, color in pillar_info:
    weight = {"pillar_possession_regain":0.40, "pillar_positional_screening":0.25,
              "pillar_counter-pressing_v2":0.20, "pillar_distribution_security":0.15}[col]
    fig_bar.add_trace(go.Bar(
        y=top_n["player_name"] + " (" + top_n["team_name"] + ")",
        x=(top_n[col] * weight).round(1),
        name=name, orientation="h", marker_color=color,
        hovertemplate="%{y}<br>" + name + ": %{x:.1f}<extra></extra>"
    ))

fig_bar.update_layout(
    barmode="stack",
    title="Top 25 Defensive Midfielders — Serie A 2025/26<br><sub>Weighted Composite Score (pillar breakdown)</sub>",
    xaxis_title="Composite Score", yaxis_title="",
    height=700, template="plotly_white",
    legend=dict(orientation="h", y=-0.12, x=0.5, xanchor="center"),
    margin=dict(l=200),
)
fig_bar.show()

## 📈 Quantity (/90) vs Quality (Percentile) — Dual View

For each KPI we plot the **per-90 value** (x-axis = how often) against the **percentile rank** (y-axis = how good relative to peers). Top-right quadrant = high quantity AND high quality.

In [9]:
# ── Quantity vs Quality scatter plots ─────────────────────────────────────────

qty_qual_kpis = [
    ("tackles_won_p90",       "tackles_won_p90_pct",       "Tackles Won"),
    ("pass_cuts_p90",         "pass_cuts_p90_pct",         "Pass Cuts (Int.+Blk.)"),
    ("ball_recoveries_p90",   "ball_recoveries_p90_pct",   "Ball Recoveries"),
    ("clearances_p90",        "clearances_p90_pct",        "Clearances"),
    ("ball_rec_opp_half_p90", "ball_rec_opp_half_p90_pct", "Ball Rec. Opp Half"),
    ("tackles_opp_half_p90",  "tackles_opp_half_p90_pct",  "Tackles Opp Half"),
    ("turnovers_p90",         "turnovers_p90_pct",         "Turnovers (inv.)"),
    ("pass_completion_pct",   "pass_completion_pct_pct",   "Pass Completion %"),
    ("short_pass_pct",        "short_pass_pct_pct",        "Short Pass %"),
]

n = len(qty_qual_kpis)
rows, cols = 3, 3
fig_qq = make_subplots(rows=rows, cols=cols,
                       subplot_titles=[t[2] for t in qty_qual_kpis],
                       horizontal_spacing=0.08, vertical_spacing=0.10)

top10_names = set(ps.head(10)["player_name"])

for i, (p90_col, pct_col, label) in enumerate(qty_qual_kpis):
    r, c = i // cols + 1, i % cols + 1
    
    others = ps[~ps["player_name"].isin(top10_names)]
    fig_qq.add_trace(go.Scatter(
        x=others[p90_col], y=others[pct_col],
        mode="markers", marker=dict(size=6, color="lightgrey", opacity=0.6),
        text=others["player_name"] + " (" + others["team_name"] + ")",
        hovertemplate="%{text}<br>" + label + " p90: %{x:.2f}<br>Percentile: %{y:.0f}<extra></extra>",
        showlegend=False,
    ), row=r, col=c)
    
    top = ps[ps["player_name"].isin(top10_names)]
    fig_qq.add_trace(go.Scatter(
        x=top[p90_col], y=top[pct_col],
        mode="markers+text", marker=dict(size=10, color="#8a1f33"),
        text=top["player_name"].str.split().str[-1],
        textposition="top center", textfont=dict(size=8),
        hovertemplate="%{customdata}<br>" + label + " p90: %{x:.2f}<br>Percentile: %{y:.0f}<extra></extra>",
        customdata=top["player_name"] + " (" + top["team_name"] + ")",
        showlegend=False,
    ), row=r, col=c)
    
    fig_qq.add_hline(y=50, line_dash="dot", line_color="grey", opacity=0.4, row=r, col=c)
    fig_qq.add_vline(x=ps[p90_col].median(), line_dash="dot", line_color="grey", opacity=0.4, row=r, col=c)

fig_qq.update_layout(
    title="Quantity (p90) vs Quality (Percentile) — All KPIs<br><sub>Top 10 composite players highlighted in red | Possession-adjusted</sub>",
    height=900, template="plotly_white", showlegend=False,
)
fig_qq.update_xaxes(title_text="Per 90", title_font_size=9)
fig_qq.update_yaxes(title_text="Percentile", title_font_size=9)
fig_qq.show()

<details><summary><b>📖 KPI Glossary</b> (click to expand)</summary>

| KPI | Definition | Pillar | Notes |
|-----|-----------|--------|-------|
| **Tackles Won p90** | Successful tackles per 90 min | Possession Regain | Possession-adjusted |
| **Pass Cuts p90** | Interceptions + blocked passes per 90 min | Possession Regain | Merged KPI; possession-adjusted |
| **Ball Recoveries p90** | Loose balls won back per 90 min | Possession Regain | Possession-adjusted |
| **Clearances p90** | Defensive clearances per 90 min | Positional Screening | Possession-adjusted |
| **Aerial Win %** | % of aerial duels won | Positional Screening | Ratio — not adjusted |
| **Ball Rec. Opp Half p90** | Recoveries in opp. half per 90 min | Counter-pressing | High-press indicator |
| **Tackles Opp Half p90** | Successful tackles in opp. half per 90 min | Counter-pressing | High-press indicator |
| **Fast Regain %** | % of regains within ≤ 8 opp. passes | Counter-pressing | Higher = better |
| **Turnovers p90 (inv.)** | Dispossessions + bad touches per 90 min | Distribution Security | Fewer = better (inverted pctl) |
| **Pass Completion %** | % of passes completed | Distribution Security | Ratio — not adjusted |
| **Short Pass %** | % of completed passes that were short | Distribution Security | Secure distribution proxy |

**Reading the chart**: x-axis = raw per-90 value (quantity), y-axis = percentile rank (quality). Top-right quadrant = high volume AND high rank among peers.

</details>

## 🕸️ Detailed Radar — Individual KPI Percentiles

Select a player from the dropdown to see their full 11-KPI radar with percentile values. League-average (50th pctl) shown as a grey dashed benchmark.

In [10]:
# ── Detailed radar — dropdown per player ───────────────────────────────────────

radar_kpis = [
    ("tackles_won_p90_pct",       "Tackles Won"),
    ("pass_cuts_p90_pct",         "Pass Cuts"),
    ("ball_recoveries_p90_pct",   "Ball Recoveries"),
    ("clearances_p90_pct",        "Clearances"),
    ("aerial_win_pct_pct",        "Aerial Win %"),
    ("ball_rec_opp_half_p90_pct", "Ball Rec. Opp Half"),
    ("tackles_opp_half_p90_pct",  "Tackles Opp Half"),
    ("fast_regain_pct_pct",       "Fast Regain %"),
    ("pass_completion_pct_pct",   "Pass Compl. %"),
    ("short_pass_pct_pct",        "Short Pass %"),
    ("turnovers_p90_pct",         "Turnovers (inv.)"),
]
radar_cols   = [c for c, _ in radar_kpis]
radar_labels = [l for _, l in radar_kpis]

fig_radar_dd = go.Figure()

avg_vals = [50] * len(radar_labels) + [50]
fig_radar_dd.add_trace(go.Scatterpolar(
    r=avg_vals, theta=radar_labels + [radar_labels[0]],
    line=dict(color="grey", width=1, dash="dot"), fill="none",
    name="League Average (50th pctl)", visible=True,
))

player_list = ps.sort_values("rank")
for i, (_, row) in enumerate(player_list.iterrows()):
    vals = [row.get(c, 50) for c in radar_cols]
    vals = [v if pd.notna(v) else 50 for v in vals]
    vals_closed = vals + [vals[0]]

    hover = []
    for pct_c, lbl in radar_kpis:
        pct_v = row.get(pct_c, np.nan)
        pct_str = f"{pct_v:.0f}" if pd.notna(pct_v) else "N/A"
        hover.append(f"{lbl}: {pct_str}th pctl")
    hover.append(hover[0])

    fig_radar_dd.add_trace(go.Scatterpolar(
        r=vals_closed, theta=radar_labels + [radar_labels[0]],
        fill="toself", fillcolor="rgba(138,31,51,0.15)",
        line=dict(color="#8a1f33", width=2.5),
        text=hover, hovertemplate="%{text}<extra></extra>",
        name=f"#{row['rank']} {row['player_name']}",
        visible=True if i == 0 else False,
    ))

buttons = []
for i, (_, row) in enumerate(player_list.iterrows()):
    vis = [True] + [j == i for j in range(len(player_list))]
    buttons.append(dict(
        label=f"#{row['rank']} {row['player_name']} ({row['team_name']})",
        method="update",
        args=[{"visible": vis},
              {"title": f"Detailed KPI Radar — #{row['rank']} {row['player_name']} ({row['team_name']})"
                        f"<br><sub>Composite: {row['composite_score']:.1f} | "
                        f"{row['matches']:.0f} GP, {row['total_minutes']:.0f} min | "
                        f"Poss: {row.get('avg_possession',50):.1f}% (adj: {row.get('adj_factor',1):.2f})</sub>"}],
    ))

fig_radar_dd.update_layout(
    title=f"Detailed KPI Radar — #1 {player_list.iloc[0]['player_name']} ({player_list.iloc[0]['team_name']})"
          f"<br><sub>Composite: {player_list.iloc[0]['composite_score']:.1f} | "
          f"{player_list.iloc[0]['matches']:.0f} GP, {player_list.iloc[0]['total_minutes']:.0f} min</sub>",
    polar=dict(radialaxis=dict(range=[0, 100], showticklabels=True, tickfont_size=8),
               angularaxis=dict(tickfont_size=9)),
    height=620, template="plotly_white",
    updatemenus=[dict(
        buttons=buttons, direction="down", showactive=True,
        x=0.02, xanchor="left", y=1.15, yanchor="top",
        bgcolor="white", bordercolor="#8a1f33",
    )],
)
fig_radar_dd.show()

<details><summary><b>📖 KPI Glossary</b> (click to expand)</summary>

| KPI | What it measures | Notes |
|-----|-----------------|-------|
| **Tackles Won** | Successful tackles per 90 min | Possession-adjusted |
| **Pass Cuts** | Interceptions + blocked passes per 90 min | Merged KPI; possession-adjusted |
| **Ball Recoveries** | Loose balls won back per 90 min | Possession-adjusted |
| **Clearances** | Defensive clearances per 90 min | Possession-adjusted |
| **Aerial Win %** | % of aerial duels won | Ratio — not adjusted |
| **Ball Rec. Opp Half** | Recoveries in opp. half per 90 min | Counter-pressing proxy |
| **Tackles Opp Half** | Successful tackles in opp. half per 90 min | Counter-pressing proxy |
| **Fast Regain %** | % of regains within ≤ 8 opp. passes | Higher = better |
| **Pass Compl. %** | % of passes completed | Ratio — not adjusted |
| **Short Pass %** | % of completed passes that were short | Secure distribution proxy |
| **Turnovers (inv.)** | Dispossessions + bad touches per 90 min | Fewer = better (inverted pctl) |

All radar axes show **percentile ranks** (0–100). Grey dashed line = 50th percentile (league average).

</details>

In [11]:
# ── Overlay radar: Top 5 comparison ───────────────────────────────────────────

top5 = ps.head(5)
colors5 = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

fig_overlay = go.Figure()
for (_, row), color in zip(top5.iterrows(), colors5):
    vals = [row.get(c, 50) for c in radar_cols]
    vals = [v if pd.notna(v) else 50 for v in vals]
    vals_closed = vals + [vals[0]]
    labels_closed = radar_labels + [radar_labels[0]]
    
    fig_overlay.add_trace(go.Scatterpolar(
        r=vals_closed, theta=labels_closed,
        fill="toself", opacity=0.15,
        line=dict(color=color, width=2.5),
        name=f"#{row['rank']} {row['player_name']} ({row['team_name']}) — {row['composite_score']:.1f}",
    ))

fig_overlay.update_polars(radialaxis=dict(range=[0, 100], showticklabels=True, tickfont_size=8))
fig_overlay.update_layout(
    title="Top 5 Overlay — Detailed KPI Radar (2025/26)",
    height=600, template="plotly_white",
    legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center"),
    polar=dict(angularaxis=dict(tickfont_size=9)),
)
fig_overlay.show()

<details><summary><b>📖 KPI Glossary</b> (click to expand)</summary>

| KPI | What it measures | Notes |
|-----|-----------------|-------|
| **Tackles Won** | Successful tackles per 90 min | Possession-adjusted |
| **Pass Cuts** | Interceptions + blocked passes per 90 min | Merged KPI; possession-adjusted |
| **Ball Recoveries** | Loose balls won back per 90 min | Possession-adjusted |
| **Clearances** | Defensive clearances per 90 min | Possession-adjusted |
| **Aerial Win %** | % of aerial duels won | Ratio — not adjusted |
| **Ball Rec. Opp Half** | Recoveries in opp. half per 90 min | Counter-pressing proxy |
| **Tackles Opp Half** | Successful tackles in opp. half per 90 min | Counter-pressing proxy |
| **Fast Regain %** | % of regains within ≤ 8 opp. passes | Higher = better |
| **Pass Compl. %** | % of passes completed | Ratio — not adjusted |
| **Short Pass %** | % of completed passes that were short | Secure distribution proxy |
| **Turnovers (inv.)** | Dispossessions + bad touches per 90 min | Fewer = better (inverted pctl) |

Overlay compares two players on the same axes. Shaded areas help visualise complementary strengths.

</details>

## 📋 Player Scouting Card — Quantity & Quality

Select a player from the dropdown to see their full KPI breakdown: raw per-90 value (quantity), percentile rank (quality), and pillar scores. Color-coded by percentile tier.

In [12]:
# ── Player scouting card — dropdown-based ──────────────────────────────────────

qty_qual_pairs = [
    ("tackles_won_p90",       "tackles_won_p90_pct",       "Tackles Won",        "Possession Regain"),
    ("pass_cuts_p90",         "pass_cuts_p90_pct",         "Pass Cuts (Int.+Blk.)", "Possession Regain"),
    ("ball_recoveries_p90",   "ball_recoveries_p90_pct",   "Ball Recoveries",    "Possession Regain"),
    ("clearances_p90",        "clearances_p90_pct",        "Clearances",         "Positional Screening"),
    ("aerial_win_pct",        "aerial_win_pct_pct",        "Aerial Win %",       "Positional Screening"),
    ("ball_rec_opp_half_p90", "ball_rec_opp_half_p90_pct", "Ball Rec. Opp Half", "Counter-pressing"),
    ("tackles_opp_half_p90",  "tackles_opp_half_p90_pct",  "Tackles Opp Half",   "Counter-pressing"),
    ("fast_regain_pct",       "fast_regain_pct_pct",       f"Fast Regain % (≤{FAST_REGAIN_THRESHOLD})", "Counter-pressing"),
    ("pass_completion_pct",   "pass_completion_pct_pct",   "Pass Completion %",  "Distribution Security"),
    ("short_pass_pct",        "short_pass_pct_pct",        "Short Pass %",       "Distribution Security"),
    ("turnovers_p90",         "turnovers_p90_pct",         "Turnovers (inv.)",   "Distribution Security"),
]

pillar_colors = {
    "Possession Regain":     "#1f77b4",
    "Positional Screening":  "#ff7f0e",
    "Counter-pressing":      "#2ca02c",
    "Distribution Security": "#d62728",
}

def pctl_bg(v):
    if pd.isna(v): return "white"
    if v >= 80: return "rgba(0,180,80,0.25)"
    if v >= 60: return "rgba(0,180,80,0.10)"
    if v >= 40: return "white"
    if v >= 20: return "rgba(255,80,80,0.10)"
    return "rgba(255,80,80,0.25)"

def build_player_table(row):
    kpi_names, raw_vals, pct_vals, pillar_names = [], [], [], []
    pct_colors = []
    for raw_c, pct_c, label, pillar in qty_qual_pairs:
        kpi_names.append(label)
        rv = row.get(raw_c, np.nan)
        pv = row.get(pct_c, np.nan)
        raw_vals.append(f"{rv:.2f}" if pd.notna(rv) else "—")
        pct_vals.append(f"{pv:.0f}" if pd.notna(pv) else "—")
        pct_colors.append(pctl_bg(pv))
        pillar_names.append(pillar)

    pillar_cell_colors = [pillar_colors.get(p, "white") for p in pillar_names]
    pillar_fill = [f"rgba({int(c[1:3],16)},{int(c[3:5],16)},{int(c[5:7],16)},0.12)"
                   if c.startswith("#") else "white" for c in pillar_cell_colors]

    return go.Table(
        header=dict(
            values=["Pillar", "KPI", "Value (/90)", "Percentile"],
            fill_color="#8a1f33", font=dict(color="white", size=11),
            align="center", height=30,
        ),
        cells=dict(
            values=[pillar_names, kpi_names, raw_vals, pct_vals],
            fill_color=[pillar_fill, ["white"]*len(kpi_names), ["white"]*len(kpi_names), pct_colors],
            align=["left", "left", "center", "center"],
            font=dict(size=11), height=26,
        ),
    )

fig_card = go.Figure()
player_list_t = ps.sort_values("rank")

for i, (_, row) in enumerate(player_list_t.iterrows()):
    tbl = build_player_table(row)
    tbl.visible = True if i == 0 else False
    fig_card.add_trace(tbl)

buttons_t = []
for i, (_, row) in enumerate(player_list_t.iterrows()):
    vis = [j == i for j in range(len(player_list_t))]
    pillar_summary = (
        f"Regain: {row['pillar_possession_regain']:.0f} | "
        f"Screen: {row['pillar_positional_screening']:.0f} | "
        f"Press: {row['pillar_counter-pressing_v2']:.0f} | "
        f"Distrib: {row['pillar_distribution_security']:.0f}"
    )
    buttons_t.append(dict(
        label=f"#{row['rank']} {row['player_name']} ({row['team_name']})",
        method="update",
        args=[{"visible": vis},
              {"title": f"Scouting Card — #{row['rank']} {row['player_name']} ({row['team_name']})<br>"
                        f"<sub>Composite: {row['composite_score']:.1f} | "
                        f"{row['matches']:.0f} GP, {row['total_minutes']:.0f} min | "
                        f"Poss: {row.get('avg_possession',50):.1f}% | {pillar_summary}</sub>"}],
    ))

first = player_list_t.iloc[0]
fig_card.update_layout(
    title=f"Scouting Card — #1 {first['player_name']} ({first['team_name']})<br>"
          f"<sub>Composite: {first['composite_score']:.1f} | "
          f"{first['matches']:.0f} GP, {first['total_minutes']:.0f} min</sub>",
    height=500, margin=dict(l=10, r=10, t=100, b=10),
    updatemenus=[dict(
        buttons=buttons_t, direction="down", showactive=True,
        x=0.02, xanchor="left", y=1.18, yanchor="top",
        bgcolor="white", bordercolor="#8a1f33",
    )],
)
fig_card.show()

<details><summary><b>📖 KPI Glossary</b> (click to expand)</summary>

| KPI | What it measures | Notes |
|-----|-----------------|-------|
| **Tackles Won** | Successful tackles per 90 min | Possession-adjusted |
| **Pass Cuts** | Interceptions + blocked passes per 90 min | Merged KPI; possession-adjusted |
| **Ball Recoveries** | Loose balls won back per 90 min | Possession-adjusted |
| **Clearances** | Defensive clearances per 90 min | Possession-adjusted |
| **Aerial Win %** | % of aerial duels won | Ratio — not adjusted |
| **Ball Rec. Opp Half** | Recoveries in opp. half per 90 min | Counter-pressing proxy |
| **Tackles Opp Half** | Successful tackles in opp. half per 90 min | Counter-pressing proxy |
| **Fast Regain %** | % of regains within ≤ 8 opp. passes | Higher = better |
| **Pass Compl. %** | % of passes completed | Ratio — not adjusted |
| **Short Pass %** | % of completed passes that were short | Secure distribution proxy |
| **Turnovers (inv.)** | Dispossessions + bad touches per 90 min | Fewer = better (inverted pctl) |

Scouting card shows the **4 pillar scores** (weighted composites) plus headline per-90 stats.

</details>

## 🎯 Interactive Player Explorer

Select any player from the dropdown to see their full detailed radar alongside the league-average benchmark.

In [13]:
# ── Player explorer with dropdown ─────────────────────────────────────────────

fig_explore = go.Figure()

avg_vals = [50] * len(radar_labels) + [50]
fig_explore.add_trace(go.Scatterpolar(
    r=avg_vals, theta=radar_labels + [radar_labels[0]],
    line=dict(color="grey", width=1, dash="dot"), fill="none",
    name="League Average (50th pctl)", visible=True,
))

player_list = ps.sort_values("rank")

for i, (_, row) in enumerate(player_list.iterrows()):
    vals = [row.get(c, 50) for c in radar_cols]
    vals = [v if pd.notna(v) else 50 for v in vals]
    vals_closed = vals + [vals[0]]
    
    raw_metrics = [c.replace("_pct","") for c, _ in radar_kpis]
    hover = []
    for (pct_c, lbl), raw_c in zip(radar_kpis, raw_metrics):
        raw_v = row.get(raw_c, np.nan)
        pct_v = row.get(pct_c, np.nan)
        raw_str = f"{raw_v:.2f}" if pd.notna(raw_v) else "N/A"
        pct_str = f"{pct_v:.0f}" if pd.notna(pct_v) else "N/A"
        hover.append(f"{lbl}: {raw_str} p90 | {pct_str}th pctl")
    hover.append(hover[0])
    
    fig_explore.add_trace(go.Scatterpolar(
        r=vals_closed, theta=radar_labels + [radar_labels[0]],
        fill="toself", fillcolor="rgba(138,31,51,0.12)",
        line=dict(color="#8a1f33", width=2.5),
        text=hover, hovertemplate="%{text}<extra></extra>",
        name=f"#{row['rank']} {row['player_name']}",
        visible=True if i == 0 else False,
    ))

buttons = []
for i, (_, row) in enumerate(player_list.iterrows()):
    vis = [True] + [j == i for j in range(len(player_list))]
    buttons.append(dict(
        label=f"#{row['rank']} {row['player_name']} ({row['team_name']})",
        method="update",
        args=[{"visible": vis}],
    ))

fig_explore.update_layout(
    title="Player Explorer — Select a DM to view detailed radar<br>"
          "<sub>Hover for per-90 (quantity) + percentile (quality) for each KPI</sub>",
    polar=dict(radialaxis=dict(range=[0, 100], showticklabels=True, tickfont_size=8),
               angularaxis=dict(tickfont_size=9)),
    height=650, template="plotly_white",
    updatemenus=[dict(
        buttons=buttons, direction="down", showactive=True,
        x=0.02, xanchor="left", y=1.15, yanchor="top",
        bgcolor="white", bordercolor="#8a1f33",
    )],
)
fig_explore.show()

<details><summary><b>📖 KPI Glossary</b> (click to expand)</summary>

| KPI | What it measures | Notes |
|-----|-----------------|-------|
| **Tackles Won** | Successful tackles per 90 min | Possession-adjusted |
| **Pass Cuts** | Interceptions + blocked passes per 90 min | Merged KPI; possession-adjusted |
| **Ball Recoveries** | Loose balls won back per 90 min | Possession-adjusted |
| **Clearances** | Defensive clearances per 90 min | Possession-adjusted |
| **Aerial Win %** | % of aerial duels won | Ratio — not adjusted |
| **Ball Rec. Opp Half** | Recoveries in opp. half per 90 min | Counter-pressing proxy |
| **Tackles Opp Half** | Successful tackles in opp. half per 90 min | Counter-pressing proxy |
| **Fast Regain %** | % of regains within ≤ 8 opp. passes | Higher = better |
| **Pass Compl. %** | % of passes completed | Ratio — not adjusted |
| **Short Pass %** | % of completed passes that were short | Secure distribution proxy |
| **Turnovers (inv.)** | Dispossessions + bad touches per 90 min | Fewer = better (inverted pctl) |

Use the dropdown to explore any player's full KPI profile.

</details>

In [15]:
# ── Fast Regain % Distribution ─────────────────────────────────────────────────

fig_fr = go.Figure()

top10_ids = set(ps.head(10)["player_id"])
fr_plot = ptr_agg.merge(
    ps[["player_name","player_id","composite_score","rank","team_name"]],
    left_on="regain_player_id", right_on="player_id", how="inner"
)
fr_plot["is_top10"] = fr_plot["regain_player_id"].isin(top10_ids)

for is_top, color, sz, name in [(False,"lightgrey",7,"Others"),(True,"#8a1f33",12,"Top 10")]:
    sub = fr_plot[fr_plot["is_top10"]==is_top]
    fig_fr.add_trace(go.Scatter(
        x=sub["n_regains"], y=sub["fast_regain_pct"],
        mode="markers+text" if is_top else "markers",
        marker=dict(size=sz, color=color, opacity=0.7 if not is_top else 1),
        text=sub["regain_player"].str.split().str[-1] if is_top else None,
        textposition="top center", textfont=dict(size=8),
        hovertemplate="%{customdata[0]} (%{customdata[1]})<br>Regains: %{x}<br>Fast Regain: %{y:.1f}%<extra></extra>",
        customdata=sub[["regain_player","team_name"]].values,
        name=name,
    ))

fig_fr.add_hline(y=fr_plot["fast_regain_pct"].median(), line_dash="dot", line_color="grey",
                 annotation_text=f"Median: {fr_plot['fast_regain_pct'].median():.1f}%")

fig_fr.update_layout(
    title=f"Fast Regain % (≤{FAST_REGAIN_THRESHOLD} opp. passes) — Serie A 2025/26<br>"
          f"<sub>X = total regain events (sample size) | Y = % that were fast regains</sub>",
    xaxis_title="Total Regain Events",
    yaxis_title=f"Fast Regain % (≤{FAST_REGAIN_THRESHOLD} passes)",
    height=500, template="plotly_white",
)
fig_fr.show()